# 00 — Crown segmentation errors and ecological forest metrics

**GitHub** `kvanoost/Musanga` → branch `v2` → `Crown/00_Crown_Ecological_Metrics.ipynb`  
**Drive workspace** `MyDrive/Colab Notebooks/Musanga/V2/Crown/`

**Research question:** Does accurate individual-crown segmentation lead to reliable estimates of tropical forest structure?

This notebook is a **read-only, CPU-only analysis**. It inventories **all Baego ROIs**, reports crown reference counts and available saved predictions by ROI, and then computes ecological metrics **only for ROIs with compatible frozen historical SAM1 predictions**.

**Scientific safeguards:** Nadine's sparse annotation inventory is for crown-level validation, *not* unbiased stand metrics. Kristof's more extensive inventory is the provisional ecological reference, subject to completeness checks. Missing ROI outputs must never be mistaken for successful zero-crown predictions. No SAM inference, model fitting or parameter tuning.

## 01 — Setup and input locations

Mount Drive, verify the geospatial inputs and identify historical SAM1 Block 53 outputs. The new V2/Crown workspace is separate from historical data and is not written to by this notebook.

In [ ]:
# 01 — Mount Drive and validate historical locations
from pathlib import Path
from google.colab import drive
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
from shapely import make_valid

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
WORKSPACE = ROOT / "V2/Crown"      # reserved for later outputs; nothing written now
DATA = ROOT / "data/Geo_Ref"
PREDICTION_ROOT = ROOT / "V2/Crown/sam1_core_centroid_halo40_v2"
RGB = DATA / "Baego_georef.tif"
CROWN_K = DATA / "Annotations/Crown_Kristof.shp"
CROWN_N = DATA / "Annotations/Crown_Nadine.shp"
ROI_PATH = DATA / "Annotations/ROI_Crowns_Kvo.shp"
MIN_REF_AREA_M2 = 5.0
METHODS = ("author_style", "nms_060")
# Optional: add VERIFIED compatible SAM1 Block53 GeoPackages for additional ROIs here.
# Key is (ROI index, processing method); never point to a different model without documenting it.
EXPECTED_ROIS = tuple(range(6))

for p in (RGB, CROWN_K, CROWN_N, ROI_PATH, PREDICTION_ROOT):
    if not p.exists():
        raise FileNotFoundError(p)
with rasterio.open(RGB) as ds:
    CRS, BOUNDS = ds.crs, ds.bounds
if CRS is None or not CRS.is_projected:
    raise ValueError("Projected raster CRS is required")
print("Baego CRS:", CRS)
print("Consistent six-ROI dataset:", PREDICTION_ROOT)
print("Workspace:", WORKSPACE)

## 02 — Reference preparation and full ROI inventory

Load the original reference annotations, repair invalid geometries **in memory**, and apply the ≥5 m² reference-crown rule. Inventory every ROI in the source ROI file, irrespective of whether a saved SAM prediction exists. ROI indices refer to **zero-based GeoPackage/shapefile row order**, consistent with the historical Block 53 experiment.

In [ ]:
# 02 — Load reference crowns and inventory ALL Baego ROIs
def polygonal(g):
    if g is None or g.is_empty:
        return Polygon()
    if not g.is_valid:
        g = make_valid(g)
    if isinstance(g, (Polygon, MultiPolygon)):
        return g
    parts = [p for p in getattr(g, "geoms", [])
             if isinstance(p, (Polygon, MultiPolygon)) and not p.is_empty]
    return unary_union(parts) if parts else Polygon()

def read_projected(path):
    g = gpd.read_file(path)
    g = g.loc[g.geometry.notna() & ~g.geometry.is_empty].copy().reset_index(drop=True)
    if g.crs is None:
        bx, rb = g.total_bounds, tuple(BOUNDS)
        if not (bx[2] >= rb[0] and bx[0] <= rb[2] and
                bx[3] >= rb[1] and bx[1] <= rb[3]):
            raise ValueError(f"No CRS and input bounds do not overlap Baego: {path}")
        print(f"WARNING: {path.name} has no CRS; raw coordinates overlap the raster. Assigning raster CRS.")
        g = g.set_crs(CRS)
    else:
        g = g.to_crs(CRS)
    g["geometry"] = gpd.GeoSeries([polygonal(x) for x in g.geometry], index=g.index, crs=CRS)
    return g.loc[~g.geometry.is_empty].reset_index(drop=True)

rois = read_projected(ROI_PATH)
kristof = read_projected(CROWN_K)
nadine = read_projected(CROWN_N)
kristof = kristof.loc[kristof.geometry.area >= MIN_REF_AREA_M2].reset_index(drop=True)
nadine = nadine.loc[nadine.geometry.area >= MIN_REF_AREA_M2].reset_index(drop=True)
ROI_IDS = tuple(range(len(rois)))

rows = []
for roi_id in ROI_IDS:
    region = rois.geometry.iloc[roi_id]
    rows.append({
        "ROI": roi_id,
        "area_ha": region.area / 1e4,
        "Kristof_centroid_crowns": int(kristof.geometry.centroid.within(region).sum()),
        "Nadine_centroid_crowns": int(nadine.geometry.centroid.within(region).sum()),
        "Kristof_intersecting": int(kristof.geometry.intersects(region).sum()),
        "Nadine_intersecting": int(nadine.geometry.intersects(region).sum())
    })
ROI_REFERENCE_AUDIT = pd.DataFrame(rows)
print(f"ALL {len(ROI_IDS)} Baego ROIs inventoried; {len(kristof)} Kristof and {len(nadine)} Nadine crowns >= {MIN_REF_AREA_M2:g} m² across Baego.")
display(ROI_REFERENCE_AUDIT.round(3))
print("Reference counts are not evidence of complete annotation coverage.")

## 03 — Load all six consistent SAM1 prediction datasets

Use the new core-centroid 40 m halo run exclusively, rather than mixing it with the historical Block 53 two-ROI archive. Whole-file counts include halo polygons and are not ROI crown densities.

In [ ]:
# 03 — Load all six compatible SAM1 outputs; fail clearly if any are missing
if tuple(ROI_IDS) != EXPECTED_ROIS:
    raise ValueError(f"Expected ROI indices {EXPECTED_ROIS}, got {ROI_IDS}")
prediction, rows = {}, []
for roi_id in ROI_IDS:
    row = {"ROI": roi_id}
    region = rois.geometry.iloc[roi_id]
    for method in METHODS:
        path = PREDICTION_ROOT / f"roi{roi_id}_{method}.gpkg"
        if not path.is_file():
            raise FileNotFoundError(f"Missing saved SAM1 output: {path}")
        g = read_projected(path)
        if len(g) and not g.geometry.intersects(region).any():
            raise ValueError(f"ROI {roi_id}, {method}: no polygons intersect ROI")
        prediction[(roi_id, method)] = g
        row[method+"_halo_total"] = len(g)
        row[method+"_centroid_in_ROI"] = int(g.geometry.centroid.within(region).sum())
        row[method+"_intersecting_ROI"] = int(g.geometry.intersects(region).sum())
    rows.append(row)
ROI_AVAILABILITY = pd.DataFrame(rows)
AVAILABLE_ROIS = ROI_IDS
MISSING_ROIS = ()
display(ROI_AVAILABILITY)
print("All six ROIs loaded. Halo totals are not stand crown counts.")


## 04 — Ecological metrics for reference areas and available predictions

Evaluate all ROIs for the reference annotations and every ROI/method combination for which a compatible prediction exists. Crown counts and mean sizes use **centroids inside the ROI**; canopy footprint uses **polygons clipped to the ROI**. Summed CPA is distinct from **union canopy cover**, and boundary conventions must be documented when interpreting results.

In [ ]:
# 04 — Metrics over ALL reference ROIs, with SAM metrics where predictions exist
def metrics_for_roi(crowns, region):
    roi_area = region.area
    if roi_area <= 0:
        raise ValueError("ROI has zero area")
    centroid_crowns = crowns.loc[crowns.geometry.centroid.within(region)]
    sizes = centroid_crowns.geometry.area.to_numpy()
    intersecting = crowns.loc[crowns.geometry.intersects(region)]
    clipped = [polygonal(g.intersection(region)) for g in intersecting.geometry]
    clipped = [g for g in clipped if not g.is_empty and g.area > 0]
    sum_area = float(sum(g.area for g in clipped))
    union_area = float(unary_union(clipped).area) if clipped else 0.0
    return {
        "roi_area_ha": roi_area / 10000,
        "n_centroid": len(centroid_crowns),
        "density_ha": len(centroid_crowns)/(roi_area/10000),
        "mean_crown_m2": float(np.mean(sizes)) if len(sizes) else np.nan,
        "median_crown_m2": float(np.median(sizes)) if len(sizes) else np.nan,
        "sum_whole_centroid_cpa_m2": float(sizes.sum()),
        "sum_clipped_cpa_ha_per_ha": sum_area/roi_area,
        "union_canopy_cover_pct": 100*union_area/roi_area,
        "overlap_excess_pct_roi": 100*(sum_area-union_area)/roi_area
    }

metric_rows = []
for roi_id in ROI_IDS:
    region = rois.geometry.iloc[roi_id]
    for label, crowns in (
        ("Kristof_reference", kristof),
        ("Nadine_sparse_NOT_STAND_REFERENCE", nadine)
    ):
        metric_rows.append({"ROI": roi_id, "dataset": label,
                            **metrics_for_roi(crowns, region)})
    for method in METHODS:
        if (roi_id, method) in prediction:
            metric_rows.append({"ROI": roi_id, "dataset": f"SAM1_{method}",
                                **metrics_for_roi(prediction[(roi_id, method)], region)})
METRIC_TABLE = pd.DataFrame(metric_rows)
print("=== Per-ROI metrics: all reference areas, available SAM1 variants ===")
display(METRIC_TABLE.round(3))
print("Nadine is NOT an exhaustive stand reference. All SAM-vs-Kristof differences are provisional.")

## 05 — Differences in ecological metrics

Compute relative differences from Kristof annotations **only where saved SAM1 predictions exist**. The resulting percentages describe agreement with the annotation inventory; they are **not validated ecological biases** until annotation completeness and crown/ROI boundary definitions are checked.

In [ ]:
# 05 — Provisional SAM-minus-Kristof differences for each available ROI/method
COLS = ("density_ha", "mean_crown_m2", "median_crown_m2",
        "sum_clipped_cpa_ha_per_ha", "union_canopy_cover_pct")
diff_rows = []
for roi_id in ROI_IDS:
    section = METRIC_TABLE.loc[METRIC_TABLE.ROI == roi_id].set_index("dataset")
    ref = section.loc["Kristof_reference"]
    for method in METHODS:
        label = f"SAM1_{method}"
        if label not in section.index:
            continue
        row = {"ROI": roi_id, "method": label}
        for col in COLS:
            reference = ref[col]
            predicted = section.loc[label, col]
            row[col + "_relative_pct"] = (
                100*(predicted-reference)/reference
                if pd.notna(reference) and reference != 0 else np.nan)
        diff_rows.append(row)
DIFFERENCES = pd.DataFrame(diff_rows)
print("=== Relative difference from Kristof annotation estimates (%) ===")
if DIFFERENCES.empty:
    print("No matching historical SAM1 predictions available.")
else:
    display(DIFFERENCES.round(1).sort_values(["ROI","method"]).reset_index(drop=True))
print("ROIs missing compatible SAM1 predictions:", MISSING_ROIS)

## 06 — Figures: ecological metric differences and crown-size distributions

Show comparisons for **all available ROI/method pairs**, without assuming exactly two ROIs. Crown-size histograms use each ROI's centroid-selected reference and predictions. Unavailable ROIs remain visible in the inventory in Section 03.

In [ ]:
# 06 — Plots expand automatically as compatible ROI predictions become available
if DIFFERENCES.empty:
    print("No plots: no saved SAM1 predictions for the surveyed ROIs.")
else:
    metric_names = {
        "density_ha": "Crown density",
        "mean_crown_m2": "Mean crown area",
        "sum_clipped_cpa_ha_per_ha": "Summed clipped CPA",
        "union_canopy_cover_pct": "Union canopy cover"
    }
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for ax, (metric, label) in zip(axes.flat, metric_names.items()):
        col = metric + "_relative_pct"
        pivot = DIFFERENCES.pivot(index="ROI", columns="method", values=col)
        pivot.plot(kind="bar", ax=ax, legend=False)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(label + " — % relative to Kristof")
        ax.set_ylabel("Relative difference (%)")
        ax.set_xlabel("ROI")
        ax.tick_params(axis="x", rotation=0)
    handles, legends = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, legends, loc="upper center", ncol=len(legends),
               bbox_to_anchor=(0.5, 1.04))
    plt.show()

    bins = np.geomspace(2, 2500, 32)
    for roi_id in AVAILABLE_ROIS:
        region = rois.geometry.iloc[roi_id]
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for label, crowns in (
            ("Kristof reference", kristof),
            ("SAM1 author", prediction[(roi_id, "author_style")]),
            ("SAM1 NMS", prediction[(roi_id, "nms_060")])
        ):
            areas = crowns.loc[crowns.geometry.centroid.within(region)].geometry.area.to_numpy()
            areas = areas[areas > 0]
            if len(areas):
                ax.hist(areas, bins=bins, histtype="step", linewidth=2,
                        label=f"{label} (n={len(areas)})")
        ax.set_xscale("log")
        ax.set_xlabel("Individual crown polygon area (m²)")
        ax.set_ylabel("Count")
        ax.set_title(f"ROI {roi_id}: crown-size distributions")
        ax.legend()
        plt.show()

print("Aggregation-scale tests are deferred until geographic coverage and reference completeness are checked.")

## 07 — Interpretation and coverage decisions

**What this notebook now does:** inventories **all** Baego ROIs, calculates reference ecological metrics for each, discovers saved SAM1 outputs dynamically, and clearly separates evaluated ROIs from missing ones.

**Before quantitative ecological conclusions:**

1. Review the Kristof inventory's spatial completeness for **each** ROI. Nadine annotations are sparse and cannot validate stand-scale density or cover.
2. Check whether historical Baego ROI indices match their intended polygons; keep the original 5 m² reference cutoff.
3. Obtain frozen SAM1 predictions for the missing ROIs using the same historical workflow. This notebook **does not** run that expensive inference or silently substitute other experimental models.
4. Only once ROI coverage is adequate, study error propagation across crown size, stand structure and 10–40 m windows, with care about boundary effects and spatial non-independence.

**Historical baseline:** SAM1 ViT-H, Luki-inspired Block 53; variants `author_style` and `nms_060`. Source results remain unchanged.